# Late Interaction Retriever
> Token-level interaction scoring for fine-grained retrieval (ColBERT-style).

| Module | `anchor.retrieval` |
|--------|--------------------|
| Key classes | `LateInteractionRetriever`, `LateInteractionScorer`, `MaxSimScorer`, `SharedSpaceRetriever`, `CrossModalEncoder` |
| Difficulty | Advanced |

Late interaction models compute fine-grained token-to-token similarities
between queries and documents, enabling more precise matching than single-vector
approaches.

**Time:** ~10 minutes

## Setup

In [ ]:
from anchor.retrieval import (
    LateInteractionRetriever,
    LateInteractionScorer,
    MaxSimScorer,
    SharedSpaceRetriever,
    CrossModalEncoder,
)
from anchor.storage import InMemoryVectorStore, InMemoryContextStore
from anchor.models import ContextItem, SourceType, QueryBundle
import random

## Mock Multi-Vector Encoder
Late interaction requires a per-token encoder that returns a matrix
(one vector per token) rather than a single vector.

In [ ]:
def mock_token_encoder(text: str) -> list[list[float]]:
    """Return a list of token-level embeddings (mock)."""
    random.seed(hash(text) % 10000)
    tokens = text.split()[:20]  # cap at 20 tokens
    dim = 32
    return [[random.random() for _ in range(dim)] for _ in tokens]

sample = mock_token_encoder("hello world test")
print(f"Tokens: 3, Vectors: {len(sample)}, Dim: {len(sample[0])}")

## MaxSim Scorer
`MaxSimScorer` computes the maximum similarity between each query token
and all document tokens, then sums the scores (ColBERT-style MaxSim).

In [ ]:
scorer = MaxSimScorer()

q_vecs = mock_token_encoder("what is Python")
d_vecs = mock_token_encoder("Python is a popular programming language")

score = scorer.score(q_vecs, d_vecs)
print(f"MaxSim score: {score:.4f}")

## Late Interaction Scorer
Wraps a token encoder and a similarity scorer into a single scoring interface.

In [ ]:
li_scorer = LateInteractionScorer(
    encoder=mock_token_encoder,
    scorer=scorer,
)

score = li_scorer.score_pair("what is Python", "Python is a popular language")
print(f"Late interaction score: {score:.4f}")

## LateInteractionRetriever
Combine late interaction scoring with a retriever.

In [ ]:
vector_store = InMemoryVectorStore()
ctx_store = InMemoryContextStore()

# Single-vector embed_fn for initial candidate retrieval
def embed_fn(text: str) -> list[float]:
    padded = text[:128].ljust(128)
    return [hash(c) % 100 / 100.0 for c in padded]

li_retriever = LateInteractionRetriever(
    vector_store=vector_store,
    context_store=ctx_store,
    embed_fn=embed_fn,
    scorer=li_scorer,
)

items = [
    ContextItem(id=f"li-{i}", content=text, source=SourceType.RETRIEVAL,
               score=0.0, priority=5, token_count=12)
    for i, text in enumerate([
        "Python is great for data analysis and scripting.",
        "Rust has a steep learning curve but excellent safety.",
        "Go routines make concurrent programming straightforward.",
        "JavaScript dominates frontend web development.",
    ])
]

li_retriever.index(items)
print(f"Indexed {len(items)} items in LateInteractionRetriever.")

In [ ]:
results = li_retriever.retrieve(QueryBundle(query_str="data scripting"), top_k=3)

print("Late interaction retrieval results:")
for i, item in enumerate(results):
    print(f"  [{i+1}] score={item.score:.4f}  {item.content[:50]}")

## SharedSpaceRetriever and CrossModalEncoder
For cross-modal retrieval (e.g., text-to-image), a `CrossModalEncoder`
projects different modalities into a shared embedding space.

In [ ]:
# Mock cross-modal encoder: maps any modality to the same vector space
cross_encoder = CrossModalEncoder(
    text_encoder=embed_fn,
    image_encoder=embed_fn,  # In practice, a vision model
)

shared_retriever = SharedSpaceRetriever(
    vector_store=InMemoryVectorStore(),
    context_store=InMemoryContextStore(),
    encoder=cross_encoder,
)

print(f"SharedSpaceRetriever type: {type(shared_retriever).__name__}")
print(f"CrossModalEncoder type: {type(cross_encoder).__name__}")

## Key Takeaways
- `MaxSimScorer` implements ColBERT-style token-level MaxSim scoring
- `LateInteractionScorer` pairs a token encoder with a similarity scorer
- `LateInteractionRetriever` uses coarse candidates + fine-grained reranking
- `CrossModalEncoder` and `SharedSpaceRetriever` enable cross-modal search
- Late interaction improves precision over single-vector retrieval

**Next:** [Rerankers](07_rerankers.ipynb)